# Семинар 1. Изображение как массив

Смотрим, что на самом деле лежит в массиве после чтения файла: форма, тип, порядок каналов, цветовые пространства, потери при resize и JPEG. В конце проецируем точки моделью камеры из лекции.

Нужны `numpy`, `pillow`, `matplotlib`, `opencv-python`. Картинка вшита в ноутбук, внешние файлы и сеть не нужны.

In [ ]:
import base64
import io

import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display
from PIL import Image

print(np.__version__, cv.__version__)

Картинка в base64, ячейка свёрнута.

In [ ]:
BUTTERFLY_JPEG_B64 = (
    "..."
    "2Q=="
)

## Чтение и форма

Картинка взята из [семинара по свёрткам](https://github.com/markblumenau/dl-course-hse-masters/blob/main/seminars/03_Image_Convolution.ipynb) курса DL.

In [ ]:
jpeg_bytes = base64.b64decode(BUTTERFLY_JPEG_B64)
image = Image.open(io.BytesIO(jpeg_bytes)).convert("RGB")
rgb = np.array(image)
h, w = rgb.shape[:2]

print("PIL size:", image.size)
print("shape:", rgb.shape, "dtype:", rgb.dtype, "min/max:", rgb.min(), rgb.max())

plt.imshow(rgb)
plt.axis("off")
plt.show()

**Что видно:** `Image.size` отдаёт `(W, H)`, а массив имеет форму `(H, W, C)`. Это одна и та же картинка, просто две разные договорённости.

Посмотрим на числа напрямую: вырежем кусок 8 на 8 на краю крыла и подпишем значения красного канала.

In [ ]:
y0, x0 = 112, 120
patch = rgb[y0 : y0 + 8, x0 : x0 + 8]

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(rgb)
axes[0].add_patch(plt.Rectangle((x0, y0), 8, 8, fill=False, ec="red", lw=2))
axes[0].set_xlim(x0 - 60, x0 + 68)
axes[0].set_ylim(y0 + 68, y0 - 60)
axes[1].imshow(patch)
for i in range(8):
    for j in range(8):
        v = patch[i, j, 0]
        axes[1].text(j, i, v, ha="center", va="center", fontsize=9, color="white" if v < 128 else "black")
axes[1].set_title("R в каждом пикселе")
for ax in axes:
    ax.axis("off")
plt.show()

In [ ]:
x, y = 400, 100
print(f"точка (x={x}, y={y}), rgb[y, x] =", rgb[y, x], " rgb[x, y] =", rgb[x, y])

plt.imshow(rgb)
plt.scatter([x], [y], c="red", marker="x", s=150, label="rgb[y, x]")
plt.scatter([y], [x], c="cyan", marker="x", s=150, label="rgb[x, y]")
plt.legend(loc="lower right")
plt.axis("off")
plt.show()

Индексируем `rgb[y, x]`: сначала строка, потом столбец. Ось `y` идёт вниз. Перепутанный индекс `rgb[x, y]` здесь не падает, а молча берёт совсем другой пиксель.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for k, (name, color) in enumerate(zip("RGB", ["red", "green", "blue"])):
    axes[k].imshow(rgb[:, :, k], cmap="gray", vmin=0, vmax=255)
    axes[k].set_title(name)
    axes[k].axis("off")
    axes[3].hist(rgb[:, :, k].ravel(), bins=64, range=(0, 255), color=color, alpha=0.5)
axes[3].set_title("гистограммы каналов")
plt.show()

**Что видно:** каждый канал это обычная одноканальная картинка. Оранжевые крылья светлые в R, средние в G и почти чёрные в B. На гистограмме у B большой пик около нуля, это крылья и жилки.

## Срез это не копия

Срез NumPy возвращает view на те же данные. Если поменять его, поменяется и исходный массив.

In [ ]:
demo = rgb.copy()
crop = demo[h // 4 : 3 * h // 4, w // 4 : 3 * w // 4]
crop_copy = demo[h // 4 : 3 * h // 4, w // 4 : 3 * w // 4].copy()
print(crop.shape, np.shares_memory(crop, demo), np.shares_memory(crop_copy, demo))

crop[:8] = (255, 0, 0)
crop[-8:] = (255, 0, 0)
crop[:, :8] = (255, 0, 0)
crop[:, -8:] = (255, 0, 0)
crop_copy[:, :, 1] = 0

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
axes[0].imshow(crop)
axes[0].set_title("рамка нарисована во view")
axes[1].imshow(demo)
axes[1].set_title("а появилась в исходном")
axes[2].imshow(crop_copy)
axes[2].set_title("копия без G, исходный не тронут")
for ax in axes:
    ax.axis("off")
plt.show()

**Что видно:** рамка, нарисованная во view, оказалась и в исходном массиве. Обнуление зелёного в копии исходник не задело. Если собираетесь что-то менять, делайте `.copy()`.

## uint8 и float

`uint8` хранит целые от 0 до 255. Арифметика в этом типе идёт по модулю 256.

In [ ]:
a = np.array([[0, 20, 240, 255]], dtype=np.uint8)
print("uint8 + 30:      ", a + np.uint8(30))
print("через int16+clip:", np.clip(a.astype(np.int16) + 30, 0, 255).astype(np.uint8))
print("cv.add:          ", cv.add(a, np.full_like(a, 30)))

bright = {
    "rgb + 80 в uint8": rgb + np.uint8(80),
    "cv.add(rgb, 80)": cv.add(rgb, np.full_like(rgb, 80)),
}
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, im) in zip(axes, bright.items()):
    ax.imshow(im)
    ax.set_title(name)
    ax.axis("off")
plt.show()

**Что видно:** 255 + 30 в NumPy даёт 29, а `cv.add` насыщает до 255. На картинке переполнение выглядит как тёмные и цветные пятна ровно в самых светлых местах: белые точки на крыльях и блики на листьях.

Дальше переходим к `float32` в диапазоне [0, 1]. Сам `astype` диапазон не меняет, делить на 255 нужно явно.

In [ ]:
print(rgb.astype(np.float32).max())

rgb_f = rgb.astype(np.float32) / 255.0
back = np.rint(rgb_f * 255).astype(np.uint8)
print(rgb_f.max(), np.array_equal(back, rgb))
print("байт uint8:", rgb.nbytes, "float32:", rgb_f.nbytes)

In [ ]:
variants = {
    "исходное": rgb_f,
    "+0.2": np.clip(rgb_f + 0.2, 0, 1),
    "*1.5": np.clip(rgb_f * 1.5, 0, 1),
}
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
for ax, (name, im) in zip(axes, variants.items()):
    ax.imshow(im)
    ax.set_title(f"{name}, насыщено {(im == 1).mean():.1%}")
    ax.axis("off")

t = np.linspace(0, 1, 256)
axes[3].plot(t, t, label="исходное")
axes[3].plot(t, np.clip(t + 0.2, 0, 1), label="+0.2")
axes[3].plot(t, np.clip(t * 1.5, 0, 1), label="*1.5")
axes[3].set_xlabel("вход")
axes[3].set_ylabel("выход")
axes[3].set_aspect("equal")
axes[3].legend()
plt.show()

**Что видно:** прибавка поднимает и чёрное, картинка становится серой. Умножение оставляет чёрное чёрным, но быстрее упирает светлые места в 1. На графике это горизонтальные участки: всё, что на них попало, уже не различить и не восстановить.

## Фильтр Байера

Сенсор в каждом пикселе меряет один цвет. Сымитируем это: оставим из RGB одно число на пиксель по шаблону RGGB и восстановим цвет обратно.

In [ ]:
mosaic = np.zeros((h, w), np.uint8)
mosaic[0::2, 0::2] = rgb[0::2, 0::2, 0]
mosaic[0::2, 1::2] = rgb[0::2, 1::2, 1]
mosaic[1::2, 0::2] = rgb[1::2, 0::2, 1]
mosaic[1::2, 1::2] = rgb[1::2, 1::2, 2]

mosaic_color = np.zeros_like(rgb)
mosaic_color[0::2, 0::2, 0] = mosaic[0::2, 0::2]
mosaic_color[0::2, 1::2, 1] = mosaic[0::2, 1::2]
mosaic_color[1::2, 0::2, 1] = mosaic[1::2, 0::2]
mosaic_color[1::2, 1::2, 2] = mosaic[1::2, 1::2]

demosaiced = cv.cvtColor(mosaic, cv.COLOR_BayerBG2RGB)
err = np.abs(demosaiced.astype(float) - rgb).mean(axis=2)
print("MAE:", err.mean())

y0, x0 = 150, 300
sl = (slice(y0, y0 + 40), slice(x0, x0 + 40))
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(mosaic_color[sl])
axes[0].set_title("что измерил сенсор")
axes[1].imshow(demosaiced[sl])
axes[1].set_title("после демозаики")
axes[2].imshow(rgb[sl])
axes[2].set_title("исходное")
im = axes[3].imshow(err, cmap="magma", vmax=40)
axes[3].add_patch(plt.Rectangle((x0, y0), 40, 40, fill=False, ec="cyan", lw=1.5))
axes[3].set_title("ошибка демозаики")
plt.colorbar(im, ax=axes[3], fraction=0.035)
for ax in axes:
    ax.axis("off")
plt.show()

**Что видно:** в каждом пикселе сенсор знает только один цвет. Две трети значений восстановлены интерполяцией, а средняя ошибка всего около трёх единиц. Карта ошибки светится только на резких границах: жилки, белые точки, контур крыла. На ровных областях демозаика почти идеальна.

Имена Bayer-кодов в OpenCV исторически сдвинуты: шаблону RGGB соответствует `COLOR_BayerBG2RGB`.

## RGB и BGR

Pillow и matplotlib живут в RGB, а `cv.imread` и `cv.imdecode` возвращают BGR.

In [ ]:
bgr = cv.imdecode(np.frombuffer(jpeg_bytes, np.uint8), cv.IMREAD_COLOR)
rgb_cv = cv.cvtColor(bgr, cv.COLOR_BGR2RGB)
print("разница декодеров:", np.abs(rgb.astype(float) - rgb_cv).mean())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(bgr)
axes[0].set_title("BGR показали как RGB")
axes[1].imshow(rgb_cv)
axes[1].set_title("после cvtColor")
for ax in axes:
    ax.axis("off")
plt.show()

**Что видно:** красное стало синим. Здесь декодеры дали одинаковые пиксели, но это не гарантировано: у Pillow и OpenCV могут быть разные сборки libjpeg.

## HSV и маски

В HSV оттенок отделён от яркости. В OpenCV для `uint8` H лежит в [0, 179], S и V в [0, 255]. Красный сидит на обоих концах шкалы H, поэтому для него нужны два интервала.

In [ ]:
hue_strip = np.zeros((40, 180, 3), np.uint8)
hue_strip[..., 0] = np.arange(180)
hue_strip[..., 1:] = 255

hsv_photo = cv.cvtColor(rgb, cv.COLOR_RGB2HSV)
fig, axes = plt.subplots(2, 3, figsize=(15, 6), gridspec_kw={"height_ratios": [1, 3]})
for ax in axes[0]:
    ax.axis("off")
axes[0, 0].imshow(cv.cvtColor(hue_strip, cv.COLOR_HSV2RGB), extent=(0, 179, 0, 20))
axes[0, 0].set_title("шкала H в OpenCV: 0..179")
axes[0, 0].axis("on")
axes[0, 0].set_yticks([])
for k, (name, cmap) in enumerate([("H", "hsv"), ("S", "gray"), ("V", "gray")]):
    axes[1, k].imshow(hsv_photo[..., k], cmap=cmap, vmin=0, vmax=179 if k == 0 else 255)
    axes[1, k].set_title(name)
    axes[1, k].axis("off")
plt.show()

**Что видно:** на крыльях H почти постоянный, около 10..20, а V сильно меняется от тени к свету. Красный находится и в самом начале шкалы, и в самом конце, поэтому для него нужны два интервала. На тёмных и серых участках H шумный: при низкой насыщенности оттенок плохо определён.

In [ ]:
swatches = np.zeros((100, 300, 3), np.uint8)
swatches[:, :100] = (230, 30, 30)
swatches[:, 100:200] = (30, 230, 30)
swatches[:, 200:] = (30, 30, 230)
hsv = cv.cvtColor(swatches, cv.COLOR_RGB2HSV)
print("HSV:", hsv[50, [50, 150, 250]])

def red_mask(hsv_img, min_s=100, min_v=60):
    m1 = cv.inRange(hsv_img, (0, min_s, min_v), (10, 255, 255))
    m2 = cv.inRange(hsv_img, (170, min_s, min_v), (179, 255, 255))
    return m1 | m2

print("red_mask нашла пикселей:", np.count_nonzero(red_mask(hsv)))
mask = cv.inRange(hsv_photo, (5, 150, 100), (25, 255, 255))
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
axes[0].imshow(swatches)
axes[1].imshow(red_mask(hsv), cmap="gray", vmin=0, vmax=255)
axes[1].set_title("red_mask на образцах")
axes[2].imshow(mask, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("H в [5, 25]")
axes[3].imshow(cv.bitwise_and(rgb, rgb, mask=mask))
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

In [ ]:
dimmed = (swatches.astype(np.float32) * 0.5).astype(np.uint8)
dimmed_hsv = cv.cvtColor(dimmed, cv.COLOR_RGB2HSV)
thresholds = np.arange(0, 256, 5)
counts = [np.count_nonzero(cv.inRange(dimmed_hsv, (40, 100, int(v)), (85, 255, 255))) for v in thresholds]
counts_orig = [np.count_nonzero(cv.inRange(hsv, (40, 100, int(v)), (85, 255, 255))) for v in thresholds]
print("H, S, V зелёного до и после:", hsv[50, 150], dimmed_hsv[50, 150])

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5), gridspec_kw={"width_ratios": [1, 2]})
axes[0].imshow(np.vstack([swatches, dimmed]))
axes[0].set_title("исходные и вдвое темнее")
axes[0].axis("off")
axes[1].plot(thresholds, counts_orig, label="исходный зелёный")
axes[1].plot(thresholds, counts, label="затемнённый зелёный")
axes[1].set_xlabel("нижний порог по V")
axes[1].set_ylabel("пикселей в маске")
axes[1].legend()
plt.show()

**Что видно:** при затемнении H и S не изменились, упала только V, вдвое, со 230 до 115. Маска по зелёному ловит оба прямоугольника, пока порог по V ниже 115, а дальше затемнённый исчезает целиком. Порог, подобранный на одной съёмке, ломается при другом свете.

## Resize и алиасинг

In [ ]:
checker = ((np.indices((256, 256)).sum(axis=0) % 2) * 255).astype(np.uint8)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(checker[:16, :16], cmap="gray", vmin=0, vmax=255)
axes[0].set_title("исходник, угол 16x16")
for ax, mode, name in zip(axes[1:], [cv.INTER_NEAREST, cv.INTER_AREA], ["nearest", "area"]):
    ax.imshow(cv.resize(checker, (47, 47), interpolation=mode), cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"{name}, 256 в 47")
for ax in axes:
    ax.axis("off")
plt.show()

**Что видно:** шахматная доска с периодом в один пиксель после nearest превращается в крупный ложный узор. `INTER_AREA` усредняет и даёт почти ровный серый, что и правильно для такой частоты.

То же на фотографии: уменьшим бабочку в 6 раз и снова растянем, чтобы было видно пиксели.

In [ ]:
small_n = cv.resize(rgb, (w // 6, h // 6), interpolation=cv.INTER_NEAREST)
small_a = cv.resize(rgb, (w // 6, h // 6), interpolation=cv.INTER_AREA)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, im, name in zip(axes, [small_n, small_a], ["nearest", "area"]):
    ax.imshow(im, interpolation="nearest")
    ax.set_title(name)
    ax.axis("off")
plt.show()

**Что видно:** у nearest жилки рвутся и белые точки то пропадают, то становятся крупными: каждый пиксель взят из одной случайной точки. У area каждый пиксель это среднее по своей области, картинка мягче и честнее.

Для масок классов всё наоборот: усреднять номера классов нельзя.

In [ ]:
labels = np.zeros((8, 8), np.uint8)
labels[:, 4:] = 4
labels[2:6, 1:3] = 2
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(labels, vmin=0, vmax=4, cmap="viridis")
axes[0].set_title("маска 8x8, классы 0, 2, 4")
for ax, (name, mode) in zip(axes[1:], [("nearest", cv.INTER_NEAREST), ("linear", cv.INTER_LINEAR)]):
    up = cv.resize(labels, (31, 31), interpolation=mode)
    ax.imshow(up, vmin=0, vmax=4, cmap="viridis")
    ax.set_title(f"{name}: классы {np.unique(up).tolist()}")
for ax in axes:
    ax.axis("off")
plt.show()

**Что видно:** после linear на границах появились классы 1 и 3, которых в разметке нет, а между 0 и 4 ещё и ложный класс 2. Маски сегментации ресайзим только nearest.

## PNG и JPEG

In [ ]:
def roundtrip(img, fmt, **kw):
    buf = io.BytesIO()
    img.save(buf, format=fmt, **kw)
    back = np.array(Image.open(io.BytesIO(buf.getvalue())).convert("RGB"))
    return len(buf.getvalue()), back

png_size, png_back = roundtrip(image, "PNG")
print("PNG байт:", png_size, "MAE:", np.abs(png_back.astype(float) - rgb).mean())

qualities = [5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100]
sizes, maes, backs = [], [], {}
for q in qualities:
    size, back = roundtrip(image, "JPEG", quality=q)
    sizes.append(size / 1024)
    maes.append(np.abs(back.astype(float) - rgb).mean())
    backs[q] = back

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(qualities, sizes, "o-")
axes[0].axhline(png_size / 1024, ls="--", c="gray", label="PNG")
axes[0].set_xlabel("quality")
axes[0].set_ylabel("КБ")
axes[0].legend()
axes[1].plot(qualities, maes, "o-")
axes[1].set_xlabel("quality")
axes[1].set_ylabel("MAE")
plt.show()

In [ ]:
y0, x0 = 150, 300
sl = (slice(y0, y0 + 48), slice(x0, x0 + 48))
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, q in zip(axes, [10, 30, 90]):
    ax.imshow(backs[q][sl], interpolation="nearest")
    ax.set_title(f"JPEG q={q}")
axes[3].imshow(png_back[sl], interpolation="nearest")
axes[3].set_title("PNG")
for ax in axes:
    ax.axis("off")
plt.show()

**Что видно:** PNG возвращает пиксели один в один. JPEG при любом качестве что-то меняет, даже при 100 ошибка не ноль. На низком качестве видны блоки 8 на 8 и цветные пятна на границах: цвет JPEG хранит с меньшим разрешением, чем яркость. Разметку и промежуточные результаты храним в PNG.

## Яркость в sRGB не линейна

После RAW-пайплайна значения закодированы гамма-кривой sRGB. Середина между чёрным и белым по свету это не 0.5 в кодах.

In [ ]:
def srgb_to_linear(x):
    x = np.asarray(x, dtype=float)
    return np.where(x <= 0.04045, x / 12.92, ((x + 0.055) / 1.055) ** 2.4)

def linear_to_srgb(x):
    x = np.asarray(x, dtype=float)
    return np.where(x <= 0.0031308, 12.92 * x, 1.055 * x ** (1 / 2.4) - 0.055)

mid_light = float(linear_to_srgb(0.5))
print("половина света в кодах sRGB:", mid_light)
print("код 0.5 по свету:", float(srgb_to_linear(0.5)))

t = np.linspace(0, 1, 256)
plt.figure(figsize=(4.5, 4.5))
plt.plot(t, srgb_to_linear(t), label="sRGB")
plt.plot(t, t, "--", c="gray", label="линейно")
plt.scatter([0.5, mid_light], [srgb_to_linear(0.5), 0.5], c=["C1", "C2"], zorder=3)
plt.xlabel("код")
plt.ylabel("свет")
plt.gca().set_aspect("equal")
plt.legend()
plt.show()

Теперь на глаз. Слева полосы 0 и 1 по два пикселя, в среднем ровно половина света. В середине код 0.5, справа код 0.735.

Браузер, проектор и экраны с масштабом 125% или 200% растягивают картинку и сглаживают её в кодах. Полосы тогда частично смешиваются в коде 0.5, и левый квадрат темнеет. Поэтому выводим картинку через HTML с `image-rendering: pixelated`: пиксели растягиваются без смешивания. Отойдите от экрана или прищурьтесь, чтобы полосы слились.

In [ ]:
def show_pixels(img, scale=1):
    buf = io.BytesIO()
    Image.fromarray(img).save(buf, format="PNG")
    src = "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()
    display(HTML(f'<img src="{src}" style="image-rendering: pixelated; image-rendering: crisp-edges; '
                 f'width: {img.shape[1] * scale}px; height: {img.shape[0] * scale}px">'))

size = 160
stripes = np.zeros((size, 3 * size))
stripes[(np.arange(size) // 2) % 2 == 0, :size] = 1
stripes[:, size : 2 * size] = 0.5
stripes[:, 2 * size :] = mid_light
show_pixels((stripes * 255).round().astype(np.uint8))

**Что видно:** при честном выводе полосы сливаются с правым квадратом, а не со средним. Если у вас всё равно ближе к среднему, дисплей или вьюер где-то масштабирует со сглаживанием.

Проверку, которая от дисплея не зависит, можно сделать в коде. Размытие это и есть то, что делает глаз издалека. Размоем полосы в кодах и в линейном свете.

In [ ]:
blur_codes = cv.GaussianBlur(stripes, (0, 0), 6)
blur_light = linear_to_srgb(cv.GaussianBlur(srgb_to_linear(stripes), (0, 0), 6))
print("полосы, размытые в кодах:", blur_codes[size // 2, size // 2].round(3))
print("полосы, размытые в свете:", blur_light[size // 2, size // 2].round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 2.6))
for ax, im, name in zip(axes, [blur_codes, blur_light], ["размытие в кодах", "размытие в линейном свете"]):
    ax.imshow(im, cmap="gray", vmin=0, vmax=1)
    ax.axvline(size, c="red", lw=0.5)
    ax.axvline(2 * size, c="red", lw=0.5)
    ax.set_title(name)
    ax.axis("off")
plt.show()

**Что видно:** если размывать коды, полосы превращаются в 0.5 и сливаются со средним квадратом. Если размывать свет, получается 0.735 и они сливаются с правым. Физически правильно второе: свет складывается линейно. Ровно эту ошибку делает любой вьюер, который масштабирует картинку в кодах, и ровно её делают `cv.resize` и `GaussianBlur` на обычных картинках.

## Модель камеры

Проецируем точки по формуле из лекции: $u = f_x X/Z + c_x$, $v = f_y Y/Z + c_y$.

In [ ]:
K = np.array([[800, 0, 320],
              [0, 800, 240],
              [0, 0, 1]], dtype=float)

def project(K, R, t, X):
    Xc = X @ R.T + t
    uvw = Xc @ K.T
    return uvw[:, :2] / uvw[:, 2:]

X = np.array([[1.0, 0.5, 4.0]])
uv = project(K, np.eye(3), np.zeros(3), X)
print(uv)

ray = np.array([[1.0, 0.5, 4.0]]) * np.array([[0.5], [1], [2], [3]])
print("точки одного луча:", project(K, np.eye(3), np.zeros(3), ray).tolist())

plt.figure(figsize=(6, 4.5))
plt.gca().add_patch(plt.Rectangle((0, 0), 640, 480, fill=False))
plt.scatter([320], [240], marker="+", c="gray", s=200, label="главная точка")
plt.scatter(*uv.T, c="red", label="(1, 0.5, 4)")
plt.xlim(-20, 660)
plt.ylim(500, -20)
plt.gca().set_aspect("equal")
plt.legend()
plt.show()

Точка (1, 0.5, 4) попадает в пиксель (520, 340). Все точки того же луча, ближе или дальше, попадают туда же: по одному пикселю глубину не восстановить. Деление на третью компоненту и есть то нелинейное место, которое матрица не убирает.

Сравним с OpenCV и добавим радиальную дисторсию. Для наглядности возьмём широкоугольную камеру с $f = 350$ px.

In [ ]:
rvec = np.zeros(3)
tvec = np.zeros(3)
uv_cv, _ = cv.projectPoints(X, rvec, tvec, K, None)
print(uv_cv.reshape(-1, 2))

K_wide = np.array([[350, 0, 320], [0, 350, 240], [0, 0, 1]], dtype=float)
gx, gy = np.meshgrid(np.linspace(-1.7, 1.7, 13), np.linspace(-1.3, 1.3, 11))
grid = np.stack([gx.ravel(), gy.ravel(), np.full(gx.size, 2.0)], axis=1)
ideal, _ = cv.projectPoints(grid, rvec, tvec, K_wide, None)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, k1, name in zip(axes, [-0.3, 0.3], ["бочка", "подушка"]):
    dist = np.array([k1, 0, 0, 0, 0])
    distorted, _ = cv.projectPoints(grid, rvec, tvec, K_wide, dist)
    ideal2, distorted2 = ideal.reshape(-1, 2), distorted.reshape(-1, 2)
    ax.quiver(*ideal2.T, *(distorted2 - ideal2).T, angles="xy", scale_units="xy", scale=1, width=0.003, color="gray")
    ax.scatter(*ideal2.T, s=8, label="без дисторсии")
    ax.scatter(*distorted2.T, s=8, label=f"k1 = {k1}")
    ax.set_xlim(-60, 700)
    ax.set_ylim(540, -60)
    ax.set_aspect("equal")
    ax.set_title(name)
    ax.legend(loc="upper right")
plt.show()

**Что видно:** при $k_1 < 0$ точки у края стягиваются к центру, при $k_1 > 0$ разъезжаются наружу. Длина серых векторов растёт с расстоянием от центра, в самом центре сдвига почти нет. Поэтому ошибки калибровки сначала вылезают по краям.

Одна и та же сцена при разных фокусных. Куб стоит в 6 м от камеры.

In [ ]:
cube = np.array([[x, y, z] for x in (-1, 1) for y in (-1, 1) for z in (5, 7)], dtype=float)
edges = [(i, j) for i in range(8) for j in range(i + 1, 8) if np.sum(cube[i] != cube[j]) == 1]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, f in zip(axes, [300, 600, 1200]):
    Kf = np.array([[f, 0, 320], [0, f, 240], [0, 0, 1]], dtype=float)
    uv = project(Kf, np.eye(3), np.zeros(3), cube)
    for i, j in edges:
        ax.plot(*uv[[i, j]].T, "k-")
    ax.set_xlim(0, 640)
    ax.set_ylim(480, 0)
    ax.set_aspect("equal")
    ax.set_title(f"f = {f} px")
plt.show()

**Что видно:** при большом $f$ куб занимает больше кадра, но отношение ближней и дальней грани то же: оно зависит от $Z$, а не от $f$. Теперь наоборот: оставим размер ближней грани в кадре постоянным, а камеру будем отодвигать и одновременно увеличивать $f$.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, d in zip(axes, [0, 5, 20]):
    f = 300 * (5 + d) / 5
    Kf = np.array([[f, 0, 320], [0, f, 240], [0, 0, 1]], dtype=float)
    uv = project(Kf, np.eye(3), np.array([0, 0, d]), cube)
    for i, j in edges:
        ax.plot(*uv[[i, j]].T, "k-")
    ax.set_xlim(0, 640)
    ax.set_ylim(480, 0)
    ax.set_aspect("equal")
    ax.set_title(f"камера дальше на {d} м, f = {f:.0f} px")
plt.show()

**Что видно:** ближняя грань везде одного размера, а дальняя догоняет её по мере того, как камера отходит. Это и есть «сжатие перспективы» длиннофокусным объективом: на самом деле его делает расстояние, а фокусное только возвращает объект к прежнему размеру в кадре.

## Resize меняет K

Если уменьшить картинку в $s$ раз, $f_x, f_y, c_x, c_y$ тоже умножаются на $s$. Crop сдвигает только $c_x, c_y$. Проверим на шахматной доске.

In [ ]:
board = np.array([[x * 0.1, y * 0.1, 0] for y in range(6) for x in range(9)], dtype=np.float32)
R, _ = cv.Rodrigues(np.array([0.3, -0.4, 0.1]))
t = np.array([-0.4, -0.25, 1.5])
uv_full = project(K, R, t, board)

s = 0.5
K_small = K.copy()
K_small[:2] *= s
uv_small = project(K_small, R, t, board)
print("resize, max |du|:", np.abs(uv_small - uv_full * s).max())

a, b = 100, 50
K_crop = K.copy()
K_crop[0, 2] -= a
K_crop[1, 2] -= b
uv_crop = project(K_crop, R, t, board)
print("crop,   max |du|:", np.abs(uv_crop - (uv_full - [a, b])).max())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
panels = [
    (uv_full, uv_full, (640, 480), "исходный кадр 640x480, K"),
    (uv_small, uv_full * s, (320, 240), "resize 0.5, K_small"),
    (uv_crop, uv_full - [a, b], (540, 430), f"crop от ({a}, {b}), K_crop"),
]
for ax, (pts, ref, (pw, ph), name) in zip(axes, panels):
    ax.add_patch(plt.Rectangle((0, 0), pw, ph, fill=False))
    ax.scatter(*ref.T, s=60, facecolors="none", edgecolors="red")
    ax.scatter(*pts.T, s=10)
    ax.set_xlim(-20, 660)
    ax.set_ylim(500, -20)
    ax.set_aspect("equal")
    ax.set_title(name)
plt.show()

**Что видно:** синие точки, спроецированные с пересчитанной K, попадают точно в красные кружки, то есть в преобразованные исходные проекции. Расхождение ноль с точностью до float. Если обучать что-то на уменьшенных кадрах и использовать K от полного разрешения, все 3D-оценки уедут.

Строго говоря, для resize через `cv.resize` центр пикселя сдвигается на полпикселя: $c' = s(c + 0.5) - 0.5$. Для измерений с субпиксельной точностью это важно.

## Источники

[OpenCV: Camera Calibration and 3D Reconstruction](https://docs.opencv.org/4.x/d9/d0c/group__calib3d.html),
[OpenCV: Changing Colorspaces](https://docs.opencv.org/4.x/df/d9d/tutorial_py_colorspaces.html),
[sRGB transfer function](https://en.wikipedia.org/wiki/SRGB),
[Brooks et al., Unprocessing Images for Learned Raw Denoising, CVPR 2019](https://arxiv.org/abs/1811.11127).